# Spam SMS classification

In [ ]:
import pandas as pd
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

In [ ]:
df = pd.read_csv('./data/spam.csv', encoding='latin-1')
df.head()

In [ ]:
df.shape

In [ ]:
df['Category'].value_counts()

Convert labels to numbers: ham = 0, spam = 1

In [ ]:
df['label'] = df['Category'].map({'ham': 0, 'spam': 1})
df.head()

Clean the text

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = ''.join(ch for ch in text if ch not in string.punctuation)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

clean_text("Free entry in 2 a wkly comp to win FA Cup final tkts!!!")

In [ ]:
df['cleaned'] = df['Message'].apply(clean_text)
df[['Message', 'cleaned']].head()

Train-test split

In [ ]:
X = df['cleaned']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

X_train_vec.shape

Train Logistic Regression

In [ ]:
model = LogisticRegression()
model.fit(X_train_vec, y_train)

In [ ]:
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

In [ ]:
confusion_matrix(y_test, y_pred)

Try on a new message

In [ ]:
sample = "Congratulations! You have won a free iPhone. Click here to claim."
sample_clean = clean_text(sample)
sample_vec = vectorizer.transform([sample_clean])
prediction = model.predict(sample_vec)[0]

print('spam' if prediction == 1 else 'ham')